In [ ]:
# Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
users = pd.read_parquet("scratch/data/ucl_stays/users-helsinki.parquet")

In [ ]:
users = users[
    (users["YY"] == 2024) &
    (users["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays = pd.read_parquet("scratch/data/ucl_stays/stays-helsinki.parquet")

In [ ]:
stays = stays[stays["in_helsinki"] == True]

In [ ]:
users[users["in_home"] == True]["user_id"].nunique()

In [ ]:
stays = stays[
    (stays["YY"] == 2024) &
    (stays["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays.sort_values(by="total_visit_frequency", ascending=False)

In [ ]:
frequency_period = (
    stays
    .groupby(["user_id", "stay_gid9"], as_index=False)
    .agg(
        frequency_period=("total_visit_frequency", "sum")
    )
)

In [ ]:
frequency_period

In [ ]:
stays_summary = (
    frequency_period
    .groupby("user_id")
    .agg(
        n_stays=("user_id", "count"),
        max_visit_frequency=("frequency_period", "max")
    )
    .reset_index()
)

In [ ]:
stays_summary.describe()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=30)
plt.xlabel("Number of stays per user")
plt.ylabel("Count")
plt.title("Distribution of number of stays per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["max_visit_frequency"], bins=30)
plt.xlabel("Maximum visit frequency")
plt.ylabel("Count")
plt.title("Distribution of maximum visit frequency per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=100)
plt.yscale("log")
plt.xlabel("Number of stays per user")
plt.ylabel("Count (log scale)")
plt.title("Distribution of number of stays per user (log scale)")
plt.show()

In [ ]:
users_home = (
    users
    .loc[users["in_home"] == True]
    .sort_values(["YY", "MM"])   # first year, then month
    .drop_duplicates(
        subset="user_id",
        keep="last"
    )[["user_id", "home_gid9", "work_gid9"]]
    .copy()
)

In [ ]:
users_home = users_home[users_home["work_gid9"].notna()].copy()
# Only those with a work_gid

In [ ]:

# 2. Merge with frequency_period
df_merged = frequency_period.merge(
    users_home,
    on="user_id",
    how="left"
)

In [ ]:
df_merged

In [ ]:
df_merged["is_home"] = (df_merged["stay_gid9"] == df_merged["home_gid9"]).astype(int)
df_merged["is_work"] = (df_merged["stay_gid9"] == df_merged["work_gid9"]).astype(int)

In [ ]:
df_merged

In [ ]:
df_merged[df_merged["user_id"]=="fffeea43-3aad-4f9c-a972-4ca57d35c9af"].sort_values("stay_gid9")

In [ ]:
df_merged.to_parquet("scratch/data/users_and_stays_3months.parquet")